In [2]:
import numpy as np
from pde_utils import correlated_gaussian_field, laplacian

# set random seed
np.random.seed(42)

# define parameters
k0 = 0.00625
k1 = 0.3125
k2 = 1
k3 = 0.0625
k4 = 0.05625
k5 = 0.0625
k6 = 0.02083
k7 = 0.001875
k8 = 0.14062*2
k9 = 0.25
k10 = 0.025
Drt = 0.08
Drd = 0.4
Df = 0.8
sigma = 0.75
s = 4 
f = 10 # dW update frequency
alpha = 1
beta = 1

size = 100    # number of cells
dt = 0.1     # time step
t_total = 1000.0  # run time
frame_int = 25    # seconds between saved frames

# initial fields
RT = 0.1 + 0.9 * np.random.rand(size, size)
RD = np.full((size, size), 0.1)
F  = np.zeros((size, size))

# initial noise
dW = correlated_gaussian_field(sigma, s, (size, size), 1.0)

def reaction(A, B, C):
    return (k0 + alpha*k1*A**3/(1 + k2*A**2))*B - (k3 + k4*(1+beta)*C)*A

# list to collect 2D frames
frames = []

n_steps      = int(t_total / dt)
save_every   = 5
dW_update    = int(f / dt)

for i in range(n_steps):
    R = reaction(RT, RD, F)

    RT = RT + dt * (R     + Drt * laplacian(RT))
    RD = RD + dt * (k5 - k6*RD - R + Drd * laplacian(RD))
    F  = F  + dt * (k7 + k8*RT**2/(1 + k9*RT**2) - k10*dW*F + Df * laplacian(F))

    # refresh noise
    if i % dW_update == 0:
        dW = correlated_gaussian_field(sigma, s, (size, size), 1.0)

    # save a frame
    if i % save_every == 0:
        frames.append(RT.copy())

# stack into a single 3D array: (n_frames, size, size)
frames_array = np.stack(frames, axis=0)

# Save
print('hello worldd')
#np.save('michaud_simulation_saved_standing.npy', frames_array)

#print(f"Saved {frames_array.shape[0]} frames→ 'michaud_simulation.npy'; array shape = {frames_array.shape}")




hello worldd


In [5]:
import numpy as np
from pde_utils import correlated_gaussian_field, laplacian

# ----------------- constant model parameters ----------------- #
k0, k1, k2 = 0.00625, 0.3125, 1
k3, k4, k5 = 0.0625, 0.05625, 0.0625
k6, k7, k8 = 0.02083, 0.001875, 0.14062 * 2
k9, k10    = 0.25, 0.025
Drt, Drd   = 0.08, 0.4
sigma, s   = 0.75, 4
f          = 10              # dW update frequency  [s]
alpha, beta = 1, 1

# grid / time settings
size       = 100             # cells per side
dt         = 0.1             # time step [s]
t_total    = 1000.0          # simulation time [s]
frame_int  = 1             # seconds between saved frames
save_every = int(frame_int / dt)
dW_update  = int(f / dt)
n_steps    = int(t_total / dt)

# ------------- reaction term (unchanged) ------------- #
def reaction(A, B, C):
    return (k0 + alpha*k1*A**3/(1 + k2*A**2))*B - (k3 + k4*(1+beta)*C)*A

# ------------- one full simulation ------------------- #
def run_simulation(Df: float, rng_seed: int = 42):
    """Run the RD model for a given fuel diffusion coefficient `Df`."""
    np.random.seed(rng_seed)

    # initial fields
    RT = 0.1 + 0.9 * np.random.rand(size, size)
    RD = np.full((size, size), 0.1)
    F  = np.zeros((size, size))
    dW = correlated_gaussian_field(sigma, s, (size, size), 1.0)

    frames = []

    for i in range(n_steps):
        R = reaction(RT, RD, F)

        RT += dt * (R + Drt * laplacian(RT))
        RD += dt * (k5 - k6*RD - R + Drd * laplacian(RD))
        F  += dt * (k7 + k8*RT**2/(1 + k9*RT**2) - k10*dW*F + Df * laplacian(F))

        if i % dW_update == 0:
            dW = correlated_gaussian_field(sigma, s, (size, size), 1.0)

        if i % save_every == 0:
            frames.append(RT.copy())

    return np.stack(frames, axis=0)   # → shape (n_frames, size, size)

# --------------- loop over Df values ------------------ #
Df_list = [0.2,0.4]   # pick any values you like
results = {}                      # Df → 3-D array

for Df in Df_list:
    print(f"Running simulation with Df = {Df}")
    frames = run_simulation(Df)
    results[Df] = frames
    # optional: save to disk
    # np.save(f"michaud_sim_Df{Df:.2f}.npy", frames)

print("Done.  Keys in `results`:", list(results))


Running simulation with Df = 0.2
Running simulation with Df = 0.4
Done.  Keys in `results`: [0.2, 0.4]
